In [6]:
import torch
import torch.nn as nn 
import pandas as pd 
import pickle
from model import ANN

In [7]:
# Load Encoders  

with open("label_encoder_gender.pkl", 'rb') as f: 
    label_encoder_gender = pickle.load(f)

with open("onehot_encoder_geo.pkl", 'rb') as f: 
    onehot_encoder_geo = pickle.load(f)

with open("Scaler.pkl", 'rb') as f: 
    Scaler = pickle.load(f)

In [8]:
# Recreate ANN Architecture

import torch.nn as nn

class ANN(nn.Module):

    def __init__(self, num_features):

        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(num_features,64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64,32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(32,1),
            nn.Sigmoid()
        )

    def forward(self,x):
        return self.model(x)

In [11]:
checkpoint = torch.load("model.pth")

model = ANN(checkpoint["num_features"])
model.load_state_dict(checkpoint['model_state']) 
model.eval()

ANN(
  (model): Sequential(
    (0): Linear(in_features=12, out_features=64, bias=True)
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=32, out_features=1, bias=True)
    (9): Sigmoid()
  )
)

In [12]:
# Example input data 
input_data = { 
    'CreditScore' : 600, 
    'Geography' : 'France', 
    'Gender' : 'Male', 
    'Age' : 40, 
    'Tenure' : 3, 
    'Balance' : 60000, 
    'NumOfProducts' : 2, 
    'HasCrCard' : 1, 
    'IsActiveMember' : 1, 
    'EstimatedSalary' : 50000
}

input_df = pd.DataFrame([input_data])

In [13]:
# onehot encode geography 

geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]])

geo_encoded_df = pd.DataFrame(geo_encoded, 
                              columns=onehot_encoder_geo.get_feature_names_out(['Geography']),
                              index=input_df.index)

input_df = pd.concat([input_df.drop(columns = ['Geography']), geo_encoded_df], axis=1) 
input_df

/Users/udit/Desktop/ANN_Classification/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [14]:
# Encode Categorical variables 
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [15]:
# Scalling the input varibles 
input_scaled = Scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [16]:
input_tensor = torch.tensor(input_scaled, dtype=torch.float32)

model.eval()

with torch.no_grad(): 
    output = model(input_tensor)
    prob = torch.sigmoid(output)
    prediction = (prob > 0.5).int()

print(f'Raw Output : {output.item()}')
print(f'Probability : {prob.item()}')
print(f'Prediction : {prediction.item()}')

Raw Output : 0.016611184924840927
Probability : 0.5041527152061462
Prediction : 1


In [17]:
if prob.item() > 0.5:
    print("the customer is likely to churn.")
else:
    print("the customer is not likely to churn.")

the customer is likely to churn.
